In [99]:
import numpy as np
import matplotlib.pyplot as plt

In [100]:
vocab = ["I drink and I know things.",
         "When you play the game of thrones, you win or you die.",
         "The true enemy won't wait out the storm, He brings the storm."]

In [101]:
for i in range(len(vocab)):
    sentence = vocab[i]
    sentence = sentence.lower()
    sentence = sentence.replace(".", "")
    sentence = sentence.replace(",", "")
    vocab[i] = sentence

for sentence in vocab:
    print(sentence)

def split_preprocess_sentence(sentence: str):
    sentence = sentence.lower()
    sentence = sentence.replace(".", "")
    sentence = sentence.replace(",", "")
    sentence = sentence.split()
    return sentence

i drink and i know things
when you play the game of thrones you win or you die
the true enemy won't wait out the storm he brings the storm


In [102]:
vocab_set = set()

for i in range(len(vocab)):
    sentence = vocab[i].split()
    vocab_set.update(sentence)

vocab_size = len(vocab_set)

print(vocab_set)
print(vocab_size)

{'he', 'drink', 'i', 'or', 'out', 'play', 'win', 'storm', 'game', 'and', 'of', 'you', 'true', 'brings', 'the', 'wait', 'enemy', 'know', 'die', "won't", 'things', 'when', 'thrones'}
23


In [103]:
encoded_dict = {list(vocab_set)[i]: i for i in range(vocab_size)}
decoded_dict = {i: list(vocab_set)[i] for i in range(vocab_size)}
print(encoded_dict)
print(decoded_dict)

{'he': 0, 'drink': 1, 'i': 2, 'or': 3, 'out': 4, 'play': 5, 'win': 6, 'storm': 7, 'game': 8, 'and': 9, 'of': 10, 'you': 11, 'true': 12, 'brings': 13, 'the': 14, 'wait': 15, 'enemy': 16, 'know': 17, 'die': 18, "won't": 19, 'things': 20, 'when': 21, 'thrones': 22}
{0: 'he', 1: 'drink', 2: 'i', 3: 'or', 4: 'out', 5: 'play', 6: 'win', 7: 'storm', 8: 'game', 9: 'and', 10: 'of', 11: 'you', 12: 'true', 13: 'brings', 14: 'the', 15: 'wait', 16: 'enemy', 17: 'know', 18: 'die', 19: "won't", 20: 'things', 21: 'when', 22: 'thrones'}


In [104]:
def encode(s: str) -> list[int]:
    s = split_preprocess_sentence(s)
    return [encoded_dict[c] for c in s]

def decode(s: list[int]) -> str:
    return " ".join([decoded_dict[c] for c in s])

print(encode("I DRINK OUT"))
print(decode(encode("I DRINK OUT")))

[2, 1, 4]
i drink out


In [105]:
merged_vocab = "\n".join(vocab)
print(merged_vocab)
data = encode(merged_vocab) 
data = np.array(data, dtype=np.float64)
print(data.shape)

i drink and i know things
when you play the game of thrones you win or you die
the true enemy won't wait out the storm he brings the storm
(30,)


In [106]:
n = int(0.9*len(data))
X_train, X_test = data[:n], data[n:]
print(f'n: {n}\nX_train: {X_train.shape}\nX_test: {X_test.shape}')

n: 27
X_train: (27,)
X_test: (3,)


In [107]:
def create_sequences(data, seq_len = 8):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+1:i+seq_len+1])
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(X_train)
print(f'{X_train_seq.shape}, {y_train_seq.shape}')

(19, 8), (19, 8)


In [108]:
def get_random_batch(X, y, batch_size):
    idx = np.random.randint(0, len(X), size=(batch_size, ))
    return X[idx], y[idx]

In [109]:
from dlfs.loss import CCE_Loss
from dlfs.optimizers import Optimizer_Adam
from dlfs.layers import EmbeddingLayer, PositionalEncoding, TransformerDecoder, LayerNorm, DenseLayer
from dlfs.activation import Softmax

batch_size = 5
block_size = 8
epochs = 500
lr = 3e-2
n_embed = 192
n_head = 6
n_layers = 3
dropout = 0.2

layers = [
    EmbeddingLayer(vocab_size, n_embed),
    PositionalEncoding(block_size, n_embed),
    TransformerDecoder(n_embed, n_head, block_size, n_layers, dropout),
    LayerNorm(n_embed),
    DenseLayer(n_embed, vocab_size),
    Softmax()
]

loss = CCE_Loss()
optimizer = Optimizer_Adam(learning_rate=lr)

for i in range(epochs + 1):

    x, y = get_random_batch(X_train_seq, y_train_seq, batch_size)
    x, y = x.astype(int), y.astype(int)

    layers[0].forward(x, training=True)

    for idx, l in enumerate(layers[1:], start=1):
        l.forward(layers[idx-1].output, training=True)

    output = layers[-1].output
    B, T, C = output.shape
    output = output.reshape(B*T, C)
    y = y.reshape(B*T)

    loss.backward(output, y)

    layers[-1].backward(loss.dinputs.reshape(B, T, C))
    for idx, layer in reversed(list(enumerate(layers[:-1]))):
        layer.backward(layers[idx + 1].dinputs)

    optimizer.pre_update_parameters()

    # Loop through all layers
    for layer in layers:

        optimizer.update_layer_parameters(layer)

    optimizer.post_update_parameters()

    if not i % 10:
        print(f'===== EPOCH : {i} ===== LOSS : {np.mean(loss.calculate(output, y))} =====')

===== EPOCH : 0 ===== LOSS : 4.254440315207676 =====
===== EPOCH : 10 ===== LOSS : 4.4340389588194515 =====
===== EPOCH : 20 ===== LOSS : 3.1749533192850925 =====
===== EPOCH : 30 ===== LOSS : 2.876705381629903 =====
===== EPOCH : 40 ===== LOSS : 2.7606099978852208 =====
===== EPOCH : 50 ===== LOSS : 2.9776234718857735 =====
===== EPOCH : 60 ===== LOSS : 2.961746903566799 =====
===== EPOCH : 70 ===== LOSS : 2.7696643700803256 =====
===== EPOCH : 80 ===== LOSS : 2.8654635758427864 =====
===== EPOCH : 90 ===== LOSS : 3.066282433541766 =====
===== EPOCH : 100 ===== LOSS : 3.014370732877581 =====
===== EPOCH : 110 ===== LOSS : 2.8847394292490436 =====
===== EPOCH : 120 ===== LOSS : 2.678605012668201 =====
===== EPOCH : 130 ===== LOSS : 2.8097906868245834 =====
===== EPOCH : 140 ===== LOSS : 2.972637567655185 =====
===== EPOCH : 150 ===== LOSS : 2.6511680785836957 =====
===== EPOCH : 160 ===== LOSS : 2.4621640970254877 =====
===== EPOCH : 170 ===== LOSS : 2.7068413137147513 =====
===== EPOC

In [110]:
def generate(idx, max_new_tokens):
        
        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            layers[0].forward(idx_cond, training=False)
            for i, l in enumerate(layers[1:], start=1):
                l.forward(layers[i-1].output, training=False)
    
            logits = layers[-1].output

            probs = logits[:, -1, :].reshape(-1)

            idx_next = np.argmax(np.random.multinomial(n=1, pvals=probs, size=1), axis=1).reshape(1, 1)

            idx = np.concatenate((idx, idx_next), axis=1)

        return idx


In [112]:
print(decode(generate(idx=np.zeros((1, 1), dtype=np.int16), max_new_tokens=50)[0].tolist()))

he out the win play the game of you die the win play the die the game of of thrones you game win or you or thrones you game enemy play play true enemy the play the game of thrones you win or you die the true enemy the true enemy
